# Fine-tune plate detector trên Kaggle (GPU miễn phí)

**Trước khi chạy (bắt buộc):**
1. **Settings > Accelerator > GPU T4 x2**.
2. **Settings > Internet > On**.
3. **+ Add Input > Upload** > chọn `models/plate_best.pt`. Đặt tên dataset là `plate-best` (khớp sẵn với ô code bên dưới).
4. Bấm vào file vừa upload trong khung Input, xem đường dẫn hiển thị dạng `/kaggle/input/datasets/<username>/plate-best/plate_best.pt` — sửa `KAGGLE_USERNAME` ở ô code đầu tiên cho khớp đúng `<username>` đó.
5. **Run → Run All**. Nếu vừa mới thêm Input mà notebook đã chạy từ trước, **Restart Session** rồi Run All lại (input mới không tự mount vào session đang chạy).

Kết quả: `/kaggle/working/plate_finetune_v1/weights/best.pt` — tải qua tab **Output** sau khi chạy xong.

Notebook đã được làm để **chạy lại (Run All) bao nhiêu lần cũng không lỗi** — các bước tải/tách dữ liệu tự bỏ qua nếu đã làm xong, và training ghi đè cùng 1 thư mục kết quả thay vì tạo bản `-2`, `-3`... mới mỗi lần.

**Thời gian ước tính**: ~1-2 tiếng cho 120 epoch trên GPU T4.

In [ ]:
KAGGLE_USERNAME = "vitanhkhc"  # đổi nếu username Kaggle của bạn khác
UPLOADED_MODEL_NAME = "plate-best"   # tên dataset bạn đã upload cho plate_best.pt
HELMET_MODEL_NAME = "helmet-best"    # tên dataset bạn đã upload cho helmet_best.pt

# Roboflow (TÙY CHỌN) — chỉ cần nếu muốn train thêm helmet bằng dataset VN riêng.
# Tự đăng ký free tại roboflow.com (email của bạn, không phải mình làm hộ),
# lấy API key ở Settings > API Key, dán vào đây. Để trống thì bỏ qua phần helmet.
ROBOFLOW_API_KEY = ""  # ví dụ: "rf_xxxxxxxxxxxx"

In [ ]:
!pip install -q ultralytics gdown roboflow

In [ ]:
from ultralytics import settings
settings.update({"raytune": False})  # tránh lỗi circular import của gói ray cài sẵn trên Kaggle

## 1. Biển số — dataset VN thật (8259 ảnh), đây là chỗ hỏng nặng nhất

In [ ]:
import os

if not os.path.exists("/kaggle/working/raw/yolo_plate_dataset"):
    !gdown "https://drive.google.com/uc?id=1KLK-DWgT3VoQH4fcTxAt2eB3sm7DGWAf" -O /kaggle/working/plate_yolo.zip
    !unzip -qo /kaggle/working/plate_yolo.zip -d /kaggle/working/raw
else:
    print("Đã có sẵn, bỏ qua tải lại.")

In [ ]:
import os, random, shutil

src = "/kaggle/working/raw/yolo_plate_dataset"
dst = "/kaggle/working/vn_plate_detect"

if os.path.exists(f"{dst}/data.yaml"):
    print("Đã tách sẵn train/val, bỏ qua.")
else:
    random.seed(42)

    imgs = [f for f in os.listdir(src) if f.lower().endswith(('.jpg', '.png'))
            and os.path.exists(os.path.join(src, os.path.splitext(f)[0] + '.txt'))]
    random.shuffle(imgs)
    n_val = max(1, int(len(imgs) * 0.1))
    val_set = set(imgs[:n_val])

    for split in ['train', 'val']:
        os.makedirs(f"{dst}/images/{split}", exist_ok=True)
        os.makedirs(f"{dst}/labels/{split}", exist_ok=True)

    for f in imgs:
        split = 'val' if f in val_set else 'train'
        stem = os.path.splitext(f)[0]
        shutil.copy(os.path.join(src, f), f"{dst}/images/{split}/{f}")
        shutil.copy(os.path.join(src, stem + '.txt'), f"{dst}/labels/{split}/{stem}.txt")

    print("total:", len(imgs), "train:", len(imgs) - n_val, "val:", n_val)

    with open(f"{dst}/data.yaml", "w") as fp:
        fp.write(f"path: {dst}\ntrain: images/train\nval: images/val\nnc: 1\nnames: ['plate']\n")

In [ ]:
from ultralytics import YOLO

base_model_path = f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{UPLOADED_MODEL_NAME}/plate_best.pt"
model = YOLO(base_model_path)
model.train(
    data="/kaggle/working/vn_plate_detect/data.yaml",
    epochs=120,
    imgsz=640,
    batch=32,
    device=0,
    project="/kaggle/working",
    name="plate_finetune_v1",
    exist_ok=True,
    patience=15,
)
print("Xong. File: /kaggle/working/plate_finetune_v1/weights/best.pt")

## 2. Mũ bảo hiểm (TÙY CHỌN — chỉ chạy nếu đã điền ROBOFLOW_API_KEY ở trên)

Model helmet hiện tại đã detect được (test thật 4/5 ảnh đúng) — không hỏng như biển số,
nên đây là cải thiện thêm chứ không bắt buộc. Dùng dataset "non-bao-hiem" (VN riêng).

In [ ]:
if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("viet-anh-khuc").project("non-bao-hiem-0kuqd")
    dataset = project.version(1).download("yolov8", location="/kaggle/working/vn_helmet")
    print("Đã tải xong:", dataset.location)
else:
    print("Bỏ qua — chưa điền ROBOFLOW_API_KEY.")

In [ ]:
if ROBOFLOW_API_KEY:
    # models/helmet_best.pt cũng cần upload qua Add Input giống plate_best.pt nếu muốn fine-tune tiếp
    helmet_model = YOLO(f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{HELMET_MODEL_NAME}/helmet_best.pt")
    helmet_model.train(
        data=f"{dataset.location}/data.yaml",
        epochs=60,
        imgsz=640,
        batch=32,
        device=0,
        project="/kaggle/working",
        name="helmet_finetune_v1",
        exist_ok=True,
        patience=10,
    )
    print("Xong. File: /kaggle/working/helmet_finetune_v1/weights/best.pt")

## 3. Tải kết quả về

Sau khi notebook chạy xong (Save Version > Save & Run All), vào tab **Output** bên phải,
tải `plate_finetune_v1/weights/best.pt` (và `helmet_finetune_v1/weights/best.pt` nếu có chạy phần 2).

**Nhớ:** test lại bằng ảnh thật từ `data/snapshots/` trước khi thay hẳn vào `models/`,
đừng tin thẳng số liệu val của dataset (đã ghi rõ lý do trong README dataset).